# KD-MoE: kiến trúc student

KD-MoE là một student nhỏ nhận **mixture (tiếng nói + nhiễu) dài 4 s, 16 kHz** và phân loại **36 lớp nhiễu**.
Student học từ một teacher lớn đóng băng (FRCRN → BEATs → linear probe), nhưng teacher **chỉ dạy router** "đây là nhóm
nào trong 4 nhóm"; 36 logit cuối được học bằng nhãn thật (cross-entropy).

Bốn ý chính của kiến trúc:

1. **Front-end lai + MaskGate**: `HybridBlock` ×2 chạy song song MHSA (toàn cục) và BiGRU (theo thời gian từng dải tần);
   sau đó `MaskGate` nhân một cổng 0.05…1 lên từng kênh của từng token. Nhánh phụ `NoiseHead` buộc các token đã qua cổng
   phải đủ thông tin để dựng lại log-mel của nhiễu (loss `L_front`).
2. **ViT** (`ViTBlock` ×2–4) trên lưới token 8 × 24 để lấy ngữ cảnh toàn cục.
3. **FreqPoolGRU**: gộp tần số → BiGRU theo thời gian → attention pooling, thành một embedding `e` cho cả clip.
4. **Soft-MoE**: `Router` (4 nhóm) + 4 `Expert`, mỗi expert xuất đủ 36 logit, trộn bằng `softmax(router)`.

## Ba cấu hình S / M / L

Khác nhau chỉ ở độ rộng và độ sâu; mọi thứ còn lại (mel 128 × 384, patch 16, 36 lớp, 4 nhóm) giống nhau.
Số tham số dưới đây là số đếm thật khi khởi tạo `KDMoEModel.from_yaml(config/kdmoe_{s,m,l}.yaml)`.

| Thông số | S | M | L |
| --- | ---: | ---: | ---: |
| `d` (độ rộng token) | 128 | 256 | 384 |
| `heads` (kích thước mỗi head) | 4 (32) | 8 (32) | 6 (64) |
| `n_hybrid` / `n_vit` | 2 / 2 | 2 / 4 | 2 / 4 |
| GRU hidden `h` (hybrid = final) | 64 | 128 | 192 |
| FFN hidden (`ffn_ratio` = 4) | 512 | 1 024 | 1 536 |
| `expert_hidden` | 128 | 256 | 384 |
| `router_hidden` | 64 | 64 | 64 |
| embedding `e` = 4·`h` | 256 | 512 | 768 |
| `drop_path_max` | 0.05 | 0.1 | 0.1 |
| **Tham số (toàn bộ)** | **1 278 552** | **6 445 912** | **14 313 304** |
| Tham số lúc suy luận (bỏ `NoiseHead`) | 1 245 528 | 6 380 120 | 14 214 744 |
| `lr` | 5e-4 | 3e-4 | 2e-4 |

Các cell sau dùng cấu hình **M** (`d = 256`, `h = 128`, `e = 512`) làm ví dụ cho shape; S và L chỉ đổi `d`, `h`, `e`.

## 1. Sơ đồ tổng thể

Đường liền là đường suy luận. Đường đứt nét xám chỉ có khi huấn luyện/đánh giá (cần nhiễu thuần làm đích cho `L_front`); đường đứt nét cam là chưng cất p_t của teacher vào router.

![Hai nhánh teacher và student](figures/kdmoe/fig_branches.png)

**Nhánh teacher** (trái) chạy offline một lần và chỉ để lại logit 36 lớp trong cache; **nhánh student** (phải) là nhánh duy nhất chạy khi suy luận. `p_t` được tính lúc train từ logit đã cache (nên quét được `T` mà không chạy lại teacher) và chỉ đi vào router.

Chi tiết nhánh student, kèm shape và số tham số:

![Sơ đồ student theo thứ tự chạy (cấu hình M)](figures/kdmoe/fig_student.png)

**Bảng luồng tensor** (cấu hình M, `B` = batch):

| # | Bước | Shape ra | Ghi chú |
| ---: | --- | --- | --- |
| 0 | waveform | `(B, 64000)` | 4 s × 16 kHz |
| 1 | LogMel | `(B, 128, 398)` | pad cuối 112 mẫu, `center=False` |
| 2 | chuẩn hoá + crop | `(B, 1, 128, 384)` | train: offset ngẫu nhiên; eval: giữa clip |
| 3 | SpecAugment | `(B, 1, 128, 384)` | chỉ train |
| 4 | PatchEmbed | `(B, 8, 24, 256)` | 192 token, mỗi token = 16 bin mel × 16 frame (160 ms) |
| 5 | HybridBlock × 2 | `(B, 8, 24, 256)` | shape giữ nguyên |
| 6 | MaskGate | `(B, 8, 24, 256)` + gate `g` cùng shape | |
| 6′ | NoiseHead (phụ) | `(B, 1, 128, 384)` | chỉ khi có nhiễu thuần |
| 7 | ViTBlock × 4 | `(B, 8, 24, 256)` | |
| 8 | LayerNorm | `(B, 8, 24, 256)` | |
| 9 | gộp tần số (mean trên trục 8) | `(B, 24, 256)` | |
| 10 | BiGRU | `(B, 24, 256)` | 2h = 256 |
| 11 | attention pooling + nối | `(B, 512)` | embedding `e` = 4h |
| 12 | Router | `(B, 4)` | `router_logits` |
| 13 | 4 Expert | `(B, 4, 36)` | mỗi expert một bộ 36 logit |
| 14 | trộn | `(B, 36)` | `logits` |

`forward` trả về `KDMoEOutput(logits, router_logits, e, gate, noise_mel, noise_target)`.

## 2. Khối 1: LogMel + SpecAugment (không có tham số học được)

| Tham số `frontend` | Giá trị |
| --- | --- |
| `sample_rate` | 16 000 |
| `n_fft` / `win_length` / `hop_length` | 512 / 400 / 160 (hop = 10 ms) |
| `n_mels`, `f_min`, `f_max` | 128, 20 Hz, 8 000 Hz |
| `power` | 2.0, `center=False` |
| `log_eps` / `norm_eps` | 1e-6 / 1e-5 |
| `num_frames` | 384 |

Các bước trong `LogMel.forward(waveform, noise=None)`:

1. Pad cuối `n_fft − win_length = 112` mẫu → 64 112 mẫu → mel-spectrogram `(B, 128, 398)`.
2. `log(mel + 1e-6)`.
3. **Chuẩn hoá theo clip**: trừ mean, chia std tính trên toàn bộ `128 × 398` của chính mixture đó.
4. **Crop 384 frame** (398 − 384 = 14 frame thừa): khi train, offset ngẫu nhiên `0…14` cho từng mẫu; khi eval, offset
   cố định `7` (giữa clip).
5. Nếu có nhiễu thuần: log-mel của nhiễu được chuẩn hoá bằng **mean/std của mixture** và cắt cùng offset. Đây là đích
   `noise_target` của `L_front`.

`SpecAugment` (chỉ khi `training`, chỉ tác dụng lên đầu vào, không đụng đích nhiễu): 2 mask tần số rộng tối đa 16 bin
và 2 mask thời gian rộng tối đa 24 frame, đặt về 0 (= giá trị trung bình sau chuẩn hoá).

> Lưu ý: với `n_fft = 512` (257 bin) và 128 mel từ 20 Hz, có **1/128 bộ lọc mel rỗng**; `torchaudio` cảnh báo điều này
> khi khởi tạo. Đầu ra của dải đó luôn là `log(1e-6)` trước chuẩn hoá.

## 3. Khối 2: PatchEmbed

`(B, 1, 128, 384)` → `(B, 8, 24, d)`.

| Thành phần | Chi tiết | Tham số |
| --- | --- | --- |
| `proj` | `Conv2d(1 → d, kernel 16, stride 16)` | `256·d + d` |
| `freq_pos` | `(8, d)`, init N(0, 0.02) | `8·d` |
| `time_pos` | `(24, d)`, init N(0, 0.02) | `24·d` |

`token[f, t] = Conv(patch[f, t]) + freq_pos[f] + time_pos[t]`: vị trí tần số và thời gian **tách riêng rồi cộng**, không
dùng một bảng `192 × d`. Tổng `289·d` tham số: **S 36 992 · M 73 984 · L 110 976**.

Lưới token: 128 bin / 16 = **8 dải tần** × 384 frame / 16 = **24 bước thời gian** = 192 token. Từ đây đến hết ViT,
tensor luôn có dạng `(B, F′=8, T′=24, d)`.

## 4. Khối 3: HybridBlock × 2

Hai nhánh đọc **cùng một đầu vào** `u = LN1(x)`, cộng vào **một** residual, rồi FFN có residual thứ hai.

![HybridBlock: hai nhánh song song, hai residual](figures/kdmoe/fig_hybrid.png)

```
u  = LN1(x)
Δ  = a_att · MHSA(u)  +  a_gru · proj(BiGRU(u))
x′ = x + DropPath(Δ)
y  = x′ + DropPath(FFN(LN2(x′)))
```

| Thành phần | Chi tiết |
| --- | --- |
| MHSA | `nn.MultiheadAttention(d, heads, dropout=0.1, batch_first=True)`; Q = K = V = 192 token; nhìn cả tần số lẫn thời gian |
| BiGRU | `GRU(d → h, bidirectional)`; mỗi dải tần là một chuỗi 24 bước, 8 chuỗi/mẫu dùng chung trọng số; đầu ra `2h` |
| `proj` | `Identity` khi `2h == d`; ngược lại `Linear(2h → d)`. Cả S, M, L đều có `2h == d` ⇒ `Identity` |
| `a_att`, `a_gru` | 2 scalar học được, init 1, **không weight decay** |
| FFN | `Linear(d → 4d) → GELU → Dropout(0.1) → Linear(4d → d) → Dropout(0.1)` |
| `hybrid_branches` | `both` (mặc định) / `attn` / `gru`: cắt một nhánh để ablation |

**Tham số mỗi HybridBlock:**

| Phần | Công thức | S | M | L |
| --- | --- | ---: | ---: | ---: |
| 2 × LayerNorm | `4d` | 512 | 1 024 | 1 536 |
| MHSA | `4d² + 4d` | 66 048 | 263 168 | 591 360 |
| BiGRU | `2 · 3 · (d·h + h² + 2h)` | 74 496 | 296 448 | 665 856 |
| FFN | `8d² + 5d` | 131 712 | 525 568 | 1 181 568 |
| `a_att`, `a_gru` | 2 | 2 | 2 | 2 |
| **Tổng 1 block** | | **272 770** | **1 086 210** | **2 440 322** |

**DropPath (stochastic depth)** tăng tuyến tính theo chỉ số khối trên **toàn bộ** `n_hybrid + n_vit` khối:
`p_i = drop_path_max · i / (n_hybrid + n_vit − 1)`. Với M (6 khối, tối đa 0.1): `0, 0.02, 0.04, 0.06, 0.08, 0.10`;
hai HybridBlock nhận `0` và `0.02`, bốn ViTBlock nhận `0.04 … 0.10`. Bỏ cả residual theo từng mẫu, chỉ khi train.

## 5. Khối 4: MaskGate và NoiseHead (nhánh phụ)

### MaskGate

```
g   = ε + (1 − ε) · sigmoid(W·h + b)        W ∈ R^{d×d},  ε = 0.05
out = h · g                                  (nhân từng kênh của từng token)
```

| Mục | Giá trị |
| --- | --- |
| Miền giá trị `g` | `[0.05, 1]` (không bao giờ tắt hẳn một kênh) |
| Khởi tạo | `b = 2.0` ⇒ `g ≈ 0.887`: cổng gần mở, khối ban đầu gần như identity |
| Tham số | `d² + d`: S 16 512 · M 65 792 · L 147 840 |
| Công tắc | `use_gate: false` ⇒ `g = 1`, bỏ khối |
| Theo dõi | `gate_mean` (trung bình `g`), `gate_low` (tỉ lệ `g < 0.2`) |

### NoiseHead

`Linear(d → 256)` áp dụng trên **từng token đã qua cổng**, mỗi token thành một patch `16 × 16`, ghép lại thành
`(B, 1, 128, 384)` = log-mel nhiễu dự đoán `noise_mel`.

- Tham số `256·d + 256`: S 33 024 · M 65 792 · L 98 560.
- Chỉ chạy khi `forward` nhận `noise` (train, và validation để đo `front`); **không nằm trên đường suy luận**.
- Gradient của `L_front` đi qua gate → HybridBlock → PatchEmbed, **không** đi vào ViT và các khối sau.
- `L_front` là L1 bất đối xứng, ước lượng thừa (`pred > target`, tức rò tiếng nói vào log-mel nhiễu) bị phạt nặng gấp
  `front_lam_over = 2`. Nhờ đó cổng học giữ phần mang thông tin nhiễu.

## 6. Khối 5: ViTBlock × n_vit + LayerNorm

Khối pre-LN chuẩn, MHSA trên cả 192 token, không có token CLS và không thêm embedding vị trí (vị trí đã cộng ở
`PatchEmbed`).

```
x′ = x  + DropPath(MHSA(LN1(x)))          # Q = K = V = 192 token
y  = x′ + DropPath(FFN(LN2(x′)))
```

| Phần | Công thức | S | M | L |
| --- | --- | ---: | ---: | ---: |
| 2 × LayerNorm | `4d` | 512 | 1 024 | 1 536 |
| MHSA | `4d² + 4d` | 66 048 | 263 168 | 591 360 |
| FFN | `8d² + 5d` | 131 712 | 525 568 | 1 181 568 |
| **Tổng 1 block** (`12d² + 13d`) | | **198 272** | **789 760** | **1 774 464** |
| Số khối `n_vit` | | 2 | 4 | 4 |
| **Tổng ViT** | | **396 544** | **3 159 040** | **7 097 856** |

Sau ViT là một `LayerNorm(d)` cuối (`2d` tham số: 256 / 512 / 768).

## 7. Khối 6: FreqPoolGRU, tạo embedding `e`

`(B, 8, 24, d)` → `(B, 4h)`.

![FreqPoolGRU: từ lưới token đến embedding e](figures/kdmoe/fig_freqpool.png)

- **Gộp tần số**: `freq_pool = mean` (mặc định, không tham số). Chế độ `attn` thêm `Linear(d → 64)` + `Linear(64 → 1)`
  để softmax theo trục tần số.
- **Attention pooling**: `s_t = wᵀ·tanh(W·seq_t)` với `W: 2h → 64`, `w: 64 → 1` (không bias); `α = softmax_t(s)`;
  `c = Σ_t α_t · seq_t`.
- **Embedding**: `e = [seq[:, −1, :h] ‖ seq[:, 0, h:] ‖ c]`, tức trạng thái cuối của GRU chiều xuôi, trạng thái cuối của
  chiều ngược (đứng ở frame 0) và vector pooling ⇒ độ dài `h + h + 2h = 4h`.

| | Công thức | S | M | L |
| --- | --- | ---: | ---: | ---: |
| BiGRU | `2·3·(d·h + h² + 2h)` | 74 496 | 296 448 | 665 856 |
| `score` | `Linear(2h → 64)` | 8 256 | 16 448 | 24 640 |
| `weight` | `Linear(64 → 1)`, không bias | 64 | 64 | 64 |
| **Tổng** | | **82 816** | **312 960** | **690 560** |
| Độ dài `e` | `4h` | 256 | 512 | 768 |

## 8. Khối 7: Router và 4 Expert

`e` được đưa **song song** vào Router và cả 4 Expert.

![Router, 4 Expert và trộn có trọng số](figures/kdmoe/fig_moe.png)

### Router

`Linear(4h → 64) → GELU → Linear(64 → 4)`, bias lớp cuối khởi tạo 0.
Tham số `(4h+1)·64 + 65·4`: **S 16 708 · M 33 092 · L 49 476**. Đầu ra `router_logits` `r ∈ R^4`.

### Expert

Mỗi expert: `Linear(4h → H) → GELU → Dropout(0.1) → Linear(H → 36)`, với `H = expert_hidden`.

- **Mỗi expert xuất đủ 36 logit** (không bị cắt theo nhóm); 4 expert xếp thành `(B, 4, 36)`.
- Mỗi expert khởi tạo bằng generator riêng (seed 0, 1, 2, 3), uniform `±1/√fan_in`, để 4 expert không giống nhau lúc đầu.
- Một expert: S 37 540 · M 140 580 · L 309 156 tham số; 4 expert: **S 150 160 · M 562 320 · L 1 236 624**.

### Trộn

```
w      = softmax(r)                         # (B, 4)
logits = Σ_g  w_g · z_g                     # z_g = logit 36 lớp của expert g, (B, 36)
```

| `inference_mode` | Khi train | Khi eval |
| --- | --- | --- |
| `weighted` (mặc định) | trộn mềm như trên | trộn mềm như trên |
| `top1` | trộn mềm | `logits = z_{argmax r}` |

Đây là **MoE mềm, dày đặc**: cả 4 expert luôn được tính (kể cả khi `top1`, vì `forward` luôn chạy đủ `(B, 4, 36)` rồi mới
chọn) và **không có loss cân bằng tải**.
Sự chuyên biệt hoá đến từ việc router bị ép theo phân phối nhóm của teacher (mục 10), không phải từ ràng buộc lên expert.

### 4 nhóm (cố định, `config/group_map.json`, ma trận `membership` 36 × 4 one-hot)

| Nhóm | Số lớp | Các lớp |
| --- | ---: | --- |
| E0 | 5 | Chime, Cowbell, Glockenspiel, Gong, Hi-hat |
| E1 | 12 | Bus, Computer keyboard, Drawer open or close, Fireworks, Gunshot or gunfire, Keys jangling, Knock, Microwave oven, Scissors, Shatter, Tearing, Writing |
| E2 | 9 | Acoustic guitar, Clarinet, Electric piano, Flute, Harmonica, Saxophone, Tambourine, Trumpet, Violin or fiddle |
| E3 | 10 | Applause, Bark, Burping or eructation, Cough, Fart, Finger snapping, Laughter, Meow, Squeak, Telephone |

`L_KD` học theo **phân phối nhóm `p_t` của teacher** (tổng xác suất teacher trong từng nhóm), không dùng nhãn nhóm của
mẫu. Nhãn nhóm (`batch["group"]`) chỉ vào `L_route_ce` (trọng số 0, đang tắt) và để báo cáo `router_acc`. Expert `g`
không bị khoá chỉ dự đoán lớp thuộc nhóm `g`.

## 9. Bảng tham số theo khối

| Khối | Số lượng | S | M | L | % của M |
| --- | ---: | ---: | ---: | ---: | ---: |
| PatchEmbed | 1 | 36 992 | 73 984 | 110 976 | 1.1 % |
| HybridBlock | × 2 | 545 540 | 2 172 420 | 4 880 644 | 33.7 % |
| MaskGate | 1 | 16 512 | 65 792 | 147 840 | 1.0 % |
| NoiseHead (chỉ train) | 1 | 33 024 | 65 792 | 98 560 | 1.0 % |
| ViTBlock | × 2 / 4 / 4 | 396 544 | 3 159 040 | 7 097 856 | 49.0 % |
| LayerNorm cuối | 1 | 256 | 512 | 768 | < 0.1 % |
| FreqPoolGRU | 1 | 82 816 | 312 960 | 690 560 | 4.9 % |
| Router | 1 | 16 708 | 33 092 | 49 476 | 0.5 % |
| Expert | × 4 | 150 160 | 562 320 | 1 236 624 | 8.7 % |
| **Tổng** | | **1 278 552** | **6 445 912** | **14 313 304** | 100 % |
| Suy luận (bỏ NoiseHead) | | 1 245 528 | 6 380 120 | 14 214 744 | |
| Học ở stage 1 (PatchEmbed + Hybrid + Gate + NoiseHead) | | 632 068 | 2 377 988 | 5 238 020 | 36.9 % |

Ở cấu hình M, ViT và HybridBlock chiếm gần 83 % tham số; toàn bộ phần MoE (Router + 4 Expert) chỉ ~9 %.

## 10. Kết nối chưng cất và các loss

Teacher chạy **một lần, ngoài student**, và được cache thành `teacher_{split}.npz` (fp16). Student chỉ đọc
`teacher_logits36` cho từng mixture.

![Hai nhánh teacher, student và các loss](figures/kdmoe/fig_branches.png)

| Loss | Công thức | Trọng số | Gradient chảy tới |
| --- | --- | ---: | --- |
| `L_CE` | cross-entropy (label smoothing 0.1) giữa `logits` 36 lớp và nhãn | 1.0 | Expert, Router (qua trọng số trộn), toàn bộ backbone |
| `L_KD` | `T² · KL(p_t ‖ softmax(r/T))`, `T = 2`, `p_t = softmax(z_teacher/T) @ membership` | 1.0 | Router và backbone trước `e` (không vào Expert) |
| `L_front` | L1 trên log-mel nhiễu, trọng số 2 nếu `pred > target`, 1 nếu ngược lại | 1.0 | NoiseHead, MaskGate, HybridBlock, PatchEmbed |
| `L_cons` | trung bình `1 − cos(e_i, e_j)` trên các cặp mixture cùng `noise_ytid` trong batch | 0.1 | backbone trước `e` |
| `L_route_ce` | `CE(router_logits, group)` | 0.0 (tắt) | Router |

```
L_total = L_CE + 1.0·L_KD + 0.1·L_cons + 1.0·L_front          (+ 0.0·L_route_ce)
```

- **Teacher chỉ dạy ở mức nhóm**: `p_t` là tổng xác suất teacher trong từng nhóm (36 → 4), nên student không phải bắt
  chước 36 logit của teacher mà chỉ học "mẫu này thuộc nhóm nào".
- **`L_cons`** ép embedding của các mixture khác nhau cùng sinh từ một clip nhiễu phải giống nhau, tức bất biến với phần
  tiếng nói và SNR. Việc này nhờ `PairedClipSampler`: mỗi batch gồm 32 "đơn vị", mỗi đơn vị là một cặp mixture cùng clip
  nhiễu (hoặc 1 mixture lẻ) ⇒ 32–64 mixture/batch, xen kẽ các nhãn để cân bằng.
- **Teacher p_t** được cross-fit theo `StratifiedGroupKFold` (5 fold, nhóm theo `noise_ytid`): mẫu train nhận logit
  out-of-fold, validation/test nhận trung bình 5 fold, để teacher không "thuộc bài" trên đúng clip nhiễu đã thấy.

### Tensor nào đi đâu

| Tensor | Sinh ra ở | Dùng ở |
| --- | --- | --- |
| `e` `(B, 4h)` | FreqPoolGRU | Router, 4 Expert, `L_cons` |
| `router_logits` `(B, 4)` | Router | trọng số trộn `softmax(r)`, `L_KD`, `L_route_ce`, `router_acc` |
| `logits` `(B, 36)` | trộn Expert | `L_CE`, mọi metric phân loại |
| `gate` `(B, 8, 24, d)` | MaskGate | theo dõi `gate_mean`, `gate_low` (không vào loss) |
| `noise_mel`, `noise_target` `(B, 1, 128, 384)` | NoiseHead, LogMel | `L_front` |
| `teacher_logits36` `(B, 36)` | cache teacher | `L_KD` (qua `membership`) |

## 11. Hai giai đoạn huấn luyện

| | Stage 1 | Stage 2 |
| --- | --- | --- |
| Mục đích | khởi động front-end tách nhiễu | huấn luyện toàn mạng |
| Khởi tạo | ngẫu nhiên | checkpoint tốt nhất của stage 1 |
| Học | PatchEmbed, HybridBlock, MaskGate, NoiseHead | tất cả tham số |
| Đóng băng | `vit.`, `final_ln.`, `pool_gru.`, `router.`, `experts.` | không |
| Loss | `L_front` | `L_total` (mục 10) |
| Chọn checkpoint | `front_score = −L_front` trên validation | `macro_acc` = trung bình theo SNR của balanced accuracy |

Thiết lập chung (`config/kdmoe_*.yaml`):

| Mục | Giá trị |
| --- | --- |
| Optimizer | AdamW, betas (0.9, 0.999), weight decay 0.05 |
| Không weight decay | tên chứa `bias`, `ln`, `norm`, `pos`, `a_att`, `a_gru` |
| Lịch lr | warmup 5 % rồi cosine về `min_lr = 1e-6` |
| Clip gradient | 1.0 |
| Độ chính xác | bf16 autocast (CUDA) |
| EMA | decay 0.999, đánh giá bằng trọng số EMA |
| Epoch / early stopping | tối đa 120, `patience` 15 (S: 13) |
| Batch | `PairedClipSampler`, 32 đơn vị ⇒ 32–64 mixture |
| Tăng cường | SpecAugment (2 mask tần số ≤ 16, 2 mask thời gian ≤ 24) + DropPath + Dropout 0.1 |
| Resume | lưu file resume mỗi epoch, sao lưu Drive |

Đánh giá chỉ trên **validation**, báo cáo 9 metrics (top1, top3, balanced accuracy, precision macro, recall macro,
macro F1, micro F1, mAP, macro AUC) cùng `router_acc`, `router_entropy`, `gate_mean`, `gate_low` và các số theo từng
mức SNR.

## 12. Suy luận

Chỉ cần waveform: `LogMel (crop giữa) → PatchEmbed → HybridBlock × 2 → MaskGate → ViTBlock × n → LN → FreqPoolGRU →
Router + 4 Expert → trộn → logits 36`.

- Bỏ: SpecAugment, NoiseHead, DropPath, Dropout; không cần nhiễu thuần và không cần teacher.
- Số tham số dùng: **S 1.25 M · M 6.38 M · L 14.2 M**.
- Mặc định `weighted` (cả 4 expert); đặt `inference_mode: top1` để chỉ lấy đầu ra của expert được router chọn (cả 4 expert vẫn chạy, nên chi phí tính toán
  không giảm).